Evan Edelstein
EN.605.645.82.SP26

In [1]:
from copy import deepcopy
from math import inf
from typing import List, Set, Dict, Tuple
import random

# Module 11 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

## Reinforcement Learning with Value Iteration

These are the same maps from Module 1 but the "physics" of the world have changed. In Module 1, the world was deterministic. When the agent moved "south", it went "south". When it moved "east", it went "east". Now, the agent only succeeds in going where it wants to go *sometimes*. There is a probability distribution over the possible states so that when the agent moves "south", there is a small probability that it will go "east", "north", or "west" instead and have to move from there.

There are a variety of ways to handle this problem. For example, if using A\* search, if the agent finds itself off the solution, you can simply calculate a new solution from where the agent ended up. Although this sounds like a really bad idea, it has actually been shown to work really well in video games that use formal planning algorithms (which we will cover later). When these algorithms were first designed, this was unthinkable. Thank you, Moore's Law!

Another approach is to use Reinforcement Learning which covers problems where there is some kind of general uncertainty in the actions. We're going to model that uncertainty a bit unrealistically here but it'll show you how the algorithm works.

As far as RL is concerned, there are a variety of options there: model-based and model-free, Value Iteration, Q-Learning and SARSA. You are going to use Value Iteration.

## The World Representation

As before, we're going to simplify the problem by working in a grid world. The symbols that form the grid have a special meaning as they specify the type of the terrain and the cost to enter a grid cell with that type of terrain:

```
token   terrain    cost 
.       plains     1
*       forest     3
^       hills      5
~       swamp      7
x       mountains  impassible
```

When you go from a plains node to a forest node it costs 3. When you go from a forest node to a plains node, it costs 1. You can think of the grid as a big graph. Each grid cell (terrain symbol) is a node and there are edges to the north, south, east and west (except at the edges).

There are quite a few differences between A\* Search and Reinforcement Learning but one of the most salient is that A\* Search returns a plan of N steps that gets us from A to Z, for example, A->C->E->G.... Reinforcement Learning, on the other hand, returns  a *policy* that tells us the best thing to do in **every state.**

For example, the policy might say that the best thing to do in A is go to C. However, we might find ourselves in D instead. But the policy covers this possibility, it might say, D->E. Trying this action might land us in C and the policy will say, C->E, etc. At least with offline learning, everything will be learned in advance (in online learning, you can only learn by doing and so you may act according to a known but suboptimal policy).

Nevertheless, if you were asked for a "best case" plan from (0, 0) to (n-1, n-1), you could (and will) be able to read it off the policy because there is a best action for every state. You will be asked to provide this in your assignment.

We have the same costs as before. Note that we've negated them this time because RL requires negative costs and positive rewards:

In [2]:
costs = {".": -1, "*": -3, "^": -5, "~": -7}
costs

{'.': -1, '*': -3, '^': -5, '~': -7}

and a list of offsets for `cardinal_moves`. You'll need to work this into your **actions**, A, parameter.

In [3]:
cardinal_moves = [(0, -1), (1, 0), (0, 1), (-1, 0)]

For Value Iteration, we require knowledge of the *transition* function, as a probability distribution.

The transition function, T, for this problem is 0.70 for the desired direction, and 0.10 each for the other possible directions. That is, if the agent selects "north" then 70% of the time, it will go "north" but 10% of the time it will go "east", 10% of the time it will go "west", and 10% of the time it will go "south". If agent is at the edge of the map, it simply bounces back to the current state.

You need to implement `value_iteration()` with the following parameters:

+ world: a `List` of `List`s of terrain (this is S from S, A, T, gamma, R)
+ costs: a `Dict` of costs by terrain (this is part of R)
+ goal: A `Tuple` of (x, y) stating the goal state.
+ reward: The reward for achieving the goal state.
+ actions: a `List` of possible actions, A, as offsets.
+ gamma: the discount rate

you will return a policy: 

`{(x1, y1): action1, (x2, y2): action2, ...}`

Remember...a policy is what to do in any state for all the states. Notice how this is different than A\* search which only returns actions to take from the start to the goal. This also explains why reinforcement learning doesn't take a `start` state.

You should also define a function `pretty_print_policy( cols, rows, policy)` that takes a policy and prints it out as a grid using "^" for up, "<" for left, "v" for down and ">" for right. Use "x" for any mountain or other impassable square. Note that it doesn't need the `world` because the policy has a move for every state. However, you do need to know how big the grid is so you can pull the values out of the `Dict` that is returned.

```
vvvvvvv
vvvvvvv
vvvvvvv
>>>>>>v
^^^>>>v
^^^>>>v
^^^>>>G
```

(Note that that policy is completely made up and only illustrative of the desired output). Please print it out exactly as requested: **NO EXTRA SPACES OR LINES**.

* If everything is otherwise the same, do you think that the path from (0,0) to the goal would be the same for both A\* Search and Q-Learning?
* What do you think if you have a map that looks like:

```
><>>^
>>>>v
>>>>v
>>>>v
>>>>G
```

has this converged? Is this a "correct" policy? What are the problems with this policy as it is?


In [4]:
def read_world(filename):
    result = []
    with open(filename) as f:
        for line in f.readlines():
            if len(line) > 0:
                result.append(list(line.strip()))
    return result

In [ ]:
TEXT2EMOJI = {".": "🌾", "*": "🌲", "^": "⛰", "~": "🐊", "x": "🌋"}
EMOJI2TEXT = {"🌾": ".", "🌲": "*", "⛰": "^", "🐊": "~", "🌋": "x"}
ACTION2EMOJI = {(0, 1): "⏬", (1, 0): "⏩", (-1, 0): "⏪", (0, -1): "⏫"}
ACTION2TEXT = {(-1, 0): "<", (1, 0): ">", (0, 1): "v", (0, -1): "^"}

In [6]:
from IPython.display import display_html


def display_emoji_grid(emoji_grid):
    """
    Display a List of Lists of emojis in a perfect grid (table) in a Jupyter Notebook.

    Parameters:
    emoji_grid (list of list of str): A 2D list containing emojis to display in a grid.
    """
    # Create HTML table
    html = '<table style="border-collapse: collapse;">'

    for row in emoji_grid:
        html += "<tr>"
        for emoji in row:
            html += f'<td style="border: none; padding: 0px; text-align: center; font-size: 1em;">{emoji}</td>'
        html += "</tr>"

    html += "</table>"

    # Display the HTML table
    display_html(html, raw=True)

In [7]:
astar_small_world = [
    ["🌾", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲"],
    ["🌾", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲"],
    ["🌾", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲"],
    ["🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌾"],
]

astar_full_world = [
    ["🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🐊", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "⛰"],
    ["🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "🌾", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "🌋", "🌋", "⛰"],
    ["🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "⛰", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "🌾", "🌾", "🌋", "🌾", "🌾", "🌾", "⛰", "🌾", "🌾", "⛰", "⛰", "🌋", "🌾"],
    ["🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "🌋", "⛰", "⛰", "🐊", "🐊", "🐊", "🐊", "🐊", "🌾", "🌋", "🌋", "🌋", "🌋", "🌾", "⛰", "🌾", "🌾", "🌾", "⛰", "🌋", "🌾"],
    ["🌾", "🌾", "🌋", "⛰", "⛰", "🌋", "🌋", "⛰", "⛰", "🐊", "🐊", "🐊", "🐊", "🐊", "🌋", "🌋", "🌲", "🌋", "🌋", "🌋", "⛰", "⛰", "🌾", "🌾", "⛰", "⛰", "🌾"],
    ["🌲", "🌾", "🌋", "🌋", "🌋", "🌋", "⛰", "⛰", "⛰", "🐊", "🐊", "🐊", "🌾", "🌾", "🌾", "🌋", "🌲", "🌲", "🌋", "🌋", "⛰", "⛰", "🌾", "⛰", "⛰", "🌾", "🌾"],
    ["🌲", "🌾", "🌲", "🌋", "🌋", "⛰", "⛰", "⛰", "🌾", "🌾", "🐊", "🌾", "🌾", "🌾", "🌾", "🌲", "🌲", "🌲", "🌲", "🌋", "🌋", "⛰", "⛰", "⛰", "🌾", "🌾", "🌾"],
    ["🌲", "🌲", "🌲", "🌋", "🌲", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "🌋", "⛰", "⛰", "🌾", "🌾", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌲", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "⛰", "⛰", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "⛰", "⛰", "🌾", "🌾", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "🌋", "🌋", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "🌋", "⛰", "🌾", "🌾", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "🌋", "🌋", "🌋", "⛰", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "🌾", "🌾", "⛰", "🌾"],
    ["🌲", "🌲", "🌲", "🌲", "🐊", "🌾", "⛰", "🌾", "🌾", "🌋", "🌋", "⛰", "⛰", "🌾", "⛰", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "🌾", "🌋", "⛰", "🌾"],
    ["🌲", "🌲", "🌲", "🐊", "🐊", "🐊", "🌋", "🌾", "⛰", "🌋", "🌋", "🌾", "🌾", "🌾", "⛰", "🌋", "🌲", "🌲", "🌲", "🌲", "🌲", "🌋", "🌋", "🌾", "🌋", "🌋", "⛰"],
    ["🌲", "🌲", "🌲", "🐊", "🐊", "🐊", "🌋", "⛰", "⛰", "🌋", "⛰", "🌾", "🐊", "🌾", "⛰", "🌋", "🌲", "🌲", "🌲", "🌾", "🌾", "🌋", "🌾", "🌾", "🌋", "🌋", "⛰"],
    ["🌲", "🌲", "🌲", "🌲", "🐊", "🐊", "🌋", "🌋", "🌋", "🌋", "🌾", "🌾", "🐊", "🌾", "⛰", "🌋", "🌋", "🌲", "🌾", "🌾", "🌋", "🌾", "🌾", "🌾", "⛰", "🌋", "⛰"],
    ["🌾", "🌲", "🌲", "🌲", "🌲", "🐊", "🐊", "🌋", "🌋", "🌾", "🌾", "🌾", "🐊", "🐊", "🌾", "⛰", "🌋", "🌲", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "⛰", "🌋", "⛰"],
    ["🌾", "🌾", "🌲", "🌲", "🌲", "🐊", "🐊", "🌋", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "⛰", "🌋", "🌋", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "🌋", "⛰", "⛰"],
    ["🌾", "🌾", "🌋", "🌲", "🌲", "🌾", "🐊", "🐊", "🐊", "🌾", "🌾", "🐊", "🌾", "🐊", "🐊", "🌾", "🌾", "🌋", "⛰", "🌾", "🌾", "🌾", "⛰", "🌋", "🌋", "⛰", "🌾"],
    ["🌾", "🌋", "🌋", "🌲", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🌾", "🌋", "⛰", "🌾", "🌾", "🌾", "⛰", "🌋", "🌾", "⛰", "🌾"],
    ["🌾", "🌋", "🌋", "🌾", "🌾", "🌾", "🌾", "🐊", "🌾", "🌾", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌋", "🌋", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "🌋", "⛰", "🌾"],
    ["🌾", "🌋", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "⛰", "🌋", "⛰", "⛰", "🌾", "🌾", "⛰", "🌋", "🌾", "🌾", "🌾", "🐊", "🐊", "🌾", "⛰", "🌋", "🌋", "🌾"],
    ["🌾", "🌋", "⛰", "⛰", "⛰", "🌾", "🌾", "🌾", "⛰", "⛰", "🌋", "🌋", "🌋", "⛰", "⛰", "🌋", "🌋", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🌾", "⛰", "🌋", "⛰"],
    ["🌾", "🌋", "⛰", "⛰", "🌋", "⛰", "🌾", "⛰", "⛰", "⛰", "🌋", "🌋", "⛰", "🌾", "🌋", "🌋", "🌾", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "⛰", "🌋", "⛰"],
    ["🌾", "🌋", "🌋", "🌋", "🌋", "🌋", "⛰", "⛰", "⛰", "🌾", "⛰", "⛰", "🌾", "🌾", "⛰", "⛰", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "⛰"],
    ["🌾", "🌋", "🌋", "🌋", "🌋", "⛰", "🌾", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🌾"],
    ["🌾", "🌾", "⛰", "⛰", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🌾"],
    ["🌾", "🌾", "⛰", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🌾", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🐊", "🌾"],
]

In [8]:
def text2emoji(grid):
    return [[TEXT2EMOJI[val] for val in row] for row in grid]

In [9]:
def emoji2text(grid):
    return [[EMOJI2TEXT[val] for val in row] for row in grid]

In [10]:
astar_small_world = emoji2text(astar_small_world)
astar_full_world = emoji2text(astar_full_world)

---

In [11]:
World = List[List[str]]
State = Tuple[int, int]
Reward = List[List[float]]
Policy = Dict[State, State | str]

<a id="get_states"></a>
## get_states

*`get_states` given a world map represented as a 2d list, return all the states in the map as list of (x,y)-coordinates* **Used By** 

* **world** World - 2d list of strings representing world map


**returns** List[State] - list of (x,y)-coordinates of all positions in world

In [12]:
def get_states(world: World) -> List[State]:
    states = []
    for r, row in enumerate(world):
        for c, _ in enumerate(row):
            states.append((c, r))
    return states

<a id="get_neighbors"></a>
## get_neighbors

*`get_neighbors`given a state as (x,y) coordinate, a list of actions as (x,y) offsets, and a world map represented as a 2d list return all valid neighbors from the state. A neighbor is valid if it inside the map's boundary and is not an impassible terrain. Optionally bounce-back from boundaries or impassible terrains by setting bounce_back to True.* **Used By** 

* **state** State - (x,y) coordinate of state
* **actions** List[State] - list of moves from state as (x,y) offsets
* **world** World - 2d list of strings representing world map
* **impassible** List[str] - list of impassible world map values 
* **bounce_back** bool - if true bounce-back from boundary or impassible terrains 


**returns** List[State] - list of (x,y)-coordinates of valid neighbors from state

In [13]:
def get_neighbors(state: State, actions: List[State], world: World, impassible: List[str] = ["x"], bounce_back: bool = False) -> List[State]:
    rows = len(world)
    cols = len(world[0])

    x, y = state
    neighbors = []
    for dx, dy in actions:
        x_prime, y_prime = x + dx, y + dy
        if 0 <= x_prime < cols and 0 <= y_prime < rows and world[y_prime][x_prime] not in impassible:
            neighbors.append((x_prime, y_prime))
        elif bounce_back:
            neighbors.append((x, y))
    return neighbors

<a id="get_action_value"></a>
## get_action_value

*`get_action_value` given a state and a neighboring state (as (x,y) coordinate), a value buffer, a list of actions as (x,y) offsets, a 2d list holding the immediate reward for each state, a world map represented as a 2d list, a transition probability and a discount rate, find the v-value as defined by the Bellman optimality equation $$ v = R[s,a] + \gamma * \sum_{s'}{T[s, a, s'] * V_{last}[s']}$$ Where R is the reward for taking action $a$ from state $s$, $\gamma$ is the discount rate, $T$ is the transition probability from $s$ to the successor state $s'$ and $V_{last}$ is the value buffer from the previous iteration.* **Used By** 

* **world** World - 2d list of strings representing world map
* **state** State - (x,y) coordinate of state
* **neighbor** State - (x,y) coordinate of neighboring state
* **actions** List[State] - list of moves from state as (x,y) offsets
* **rewards** Reward  - 2d list storing the immediate reward value for each state in the world
* **v_last** Reward - previous buffer of potential reward values for each state
* **transition** float - probability of taking an action
* **gamma** float - discount rate


**returns** float- the action value for taking this action

In [14]:
def get_action_value(
    world: World,
    state: State,
    neighbor: State,
    actions: List[State],
    rewards: Reward,
    v_last: Reward,
    transition: float = 1.0,
    gamma: float = 1.0,
) -> float:
    x_prime, y_prime = neighbor[0], neighbor[1]
    future_reward = transition * v_last[y_prime][x_prime]

    if transition < 1.0:
        other_transition = (1 - transition) / (len(actions) - 1)
        for x_prob, y_prob in get_neighbors(state, actions, world, bounce_back=True):
            if (x_prob, y_prob) == (x_prime, y_prime):
                continue
            future_reward += other_transition * v_last[y_prob][x_prob]

    val = rewards[y_prime][x_prime] + gamma * future_reward
    return val

<a id="get_best_actions"></a>
## get_best_actions

*`get_best_actions` given a state as an (x,y) coordinate, a value buffer, a list of actions as (x,y) offsets, a 2d list holding the immediate reward for each state, a world map represented as a 2d list, a transition probability and a discount rate, find the action that has the highest v-value as defined by the Bellman optimality equation. Ties are broken by order in actions list* **Used By** 

* **world** World - 2d list of strings representing world map
* **state** State - (x,y) coordinate of state
* **actions** List[State] - list of moves from state as (x,y) offsets
* **rewards** Reward  - 2d list storing the immediate reward value for each state in the world
* **v_last** Reward - previous buffer of potential reward values for each state
* **transition** float - probability of taking an action
* **gamma** float - discount rate

**returns** Tuple[List[State] | None, float] - Tuple containing the best action or None if no action could be found, and maximum reward value  

In [15]:
def get_best_actions(
    world: World,
    state: State,
    actions: List[State],
    rewards: Reward,
    v_last: Reward,
    transition: float = 1.0,
    gamma: float = 1.0,
) -> Tuple[State | None, float]:

    max_action, max_val = None, -inf

    for neighbor in get_neighbors(state, actions, world):
        action = neighbor[0] - state[0], neighbor[1] - state[1]
        val = get_action_value(world, state, neighbor, actions, rewards, v_last, transition, gamma)
        if val > max_val:
            max_val, max_action = val, action

    return max_action, max_val

<a id="update_policy"></a>
## update_policy

*`update_policy` given a value buffer, a goals dict, a list of actions as (x,y) offsets, a 2d list holding the immediate reward for each state, a world map represented as a 2d list, a previous policy, a transition probability and a discount rate, update a policy using value iteration by finding the best action in terms of Bellman optimality value for each state in the world. Terminal states, such as goal states and impassible states are added to the policy as "G"/"X", respectively.* **Used By** 

* **world** World - 2d list of strings representing world map
* **actions** List[State] - list of moves from state as (x,y) offsets
* **rewards** Reward  - 2d list storing the immediate reward value for each state in the world
* **goals** Dict[State, float] - dict mapping (x,y) coordinate of goal to reward value
* **v** Reward - 2d list of state reward values
* **policy** Policy - dict mapping each state to a preferred action or terminal state, such as impassible terrain or goal state
* **transition** float - probability of taking an action
* **gamma** float - discount rate
* **impassible** List[str] - list of impassible world map values 


**returns** Tuple[Policy, Reward, Reward] - tuple containing the updated policy, v-value buffer and a copy of the previous v-value buffer

In [16]:
def update_policy(
    world: World,
    actions: List[State],
    rewards: Reward,
    goals: Dict[State, float],
    v: Reward,
    policy: Policy,
    transition: float = 1.0,
    gamma: float = 1.0,
    impassible: List[str] = ["x"],
) -> Tuple[Policy, Reward, Reward]:
    v_last = deepcopy(v)

    for x, y in get_states(world):
        if world[y][x] in impassible:
            policy[(x, y)] = "X"
            continue

        if (x, y) in goals:
            v[y][x] = goals[(x, y)]
            policy[(x, y)] = "G"
            continue

        max_action, max_val = get_best_actions(world, (x, y), actions, rewards, v_last, transition, gamma)
        if max_action is None:
            continue

        policy[(x, y)] = max_action
        v[y][x] = max_val
    return policy, v, v_last

<a id="pretty_print_policy"></a>
## pretty_print_policy

*`pretty_print_policy` print a policy as a 2d map*

* **cols** int - number of cols in the world
* **rows** int - number of rows in the world
* **policy** Policy - dict mapping each state to a preferred action or terminal state, such as impassible terrain or goal state

**returns**

In [17]:
def pretty_print_policy(cols: int, rows: int, policy: Policy):
    for row in range(rows):
        for col in range(cols):
            action = policy[(col, row)]
            tile = ACTION2TEXT[action] if isinstance(action, tuple) else action
            print(tile, end="")
        print()

<a id="value_iteration"></a>
## value_iteration

*`value_iteration` .*

* **world** World - 2d list of strings representing world map
* **costs** Dict[str, int] - map of each terrain to negative reward 
* **goals** Dict[State, float] - dict mapping (x,y) coordinate of goal to reward value
* **actions** List[State] - list of moves from state as (x,y) offsets
* **transition** float - probability of taking an action
* **gamma** float - discount rate
* **e** float - epsilon threshold
* **max_iters** int - maximum number of iteration to perform, set to -1 to bypass 
* **debug** bool - if true print debug information after each iteration


**return** Policy - policy dict mapping each state to a preferred action or terminal state, such as impassible terrain or goal state

In [18]:
def value_iteration(
    world: World,
    costs: Dict[str, int],
    goals: Dict[State, float],
    actions: List[State],
    gamma: float = 1.0,
    transition: float = 1.0,
    e: float = 0.01,
    max_iters: int = 1000,
    debug: bool = False,
) -> Policy:
    rows, cols = len(world), len(world[0])

    v = [[0.0] * cols for _ in range(rows)]
    policy: Policy = {state: "G" for state in goals}
    rewards = [[float(costs.get(x, 0.0)) for x in row] for row in world]
    for (x, y), r in goals.items():
        rewards[y][x] = r

    t = 0
    while max_iters < 1 or t < max_iters:
        policy, v, v_last = update_policy(world, actions, rewards, goals, v, policy, transition, gamma)
        delta = max([abs(v[y][x] - v_last[y][x]) for x, y in get_states(world)])
        print(t, v_last, v, policy, delta) if debug else None
        if delta < e:
            return policy
        t += 1
    return policy

## Value Iteration

### Small World

In [19]:
small_world = read_world("small-2.txt")

In [20]:
goal = {(len(small_world[0]) - 1, len(small_world) - 1): 100.0}
gamma = 0.9

small_policy = value_iteration(small_world, costs, goal, cardinal_moves, gamma, transition=0.7)

In [21]:
cols = len(small_world[0])
rows = len(small_world)

pretty_print_policy(cols, rows, small_policy)

v>>>>v
vvv>vv
vvv>vv
vvvXvv
vvvvvv
>>>>vv
>>>>>G


### Large World

In [22]:
large_world = read_world("large-2.txt")

In [23]:
goal = {(len(large_world[0]) - 1, len(large_world) - 1): 1000000.0}
gamma = 0.9

large_policy = value_iteration(large_world, costs, goal, cardinal_moves, gamma, transition=0.7)

In [24]:
cols = len(large_world[0])
rows = len(large_world)

pretty_print_policy(cols, rows, large_policy)

v>>>>>>>>>>>>>>vv>>>>>>>>vv
vv>>>>>>>>>>>>vvv<XXXXXXXvv
vvv^XX>>>>>>>>>vvXXXvvvXXvv
vvvv<XXX>>>>>>>>>>>vvv<XXvv
vvvv<XXv>>>>>>>>>v>vvvXXXvv
vvv<XXvvv>>>>>>>>v>>vvvXvvv
vvvXXvvvvv>^XXX>>v>>>>>vvvv
v>>>>vvvvvv^<<XXX>>>>>vvvvv
vv>>vvvvvvv<<<XX>>>>>>>vvvv
vv>>vvvvvv<XXXX>>>>>>>>vvvv
v>>>>vvvv<XXX>>>>>vvXXXvvvv
>>>>>vvvvXXv>>>>>>>vvXXvvvv
>>>>>>vvvXXv>>>>>>>>vX>vvvv
>>>>>>v>>>vv>>>>>>>>>>>vvvv
vv>^X>v>>vvv<>>>>>>>>^Xvvvv
vv<XXX>>>>vvXXX>>>>>^XXvvvv
vvXX>>>>>>>>>vXXX>^XXXvvvvv
vvvXX>>>>>>>>>>vXXXX>>vvvvv
vvvXXX>>>>>>>>>>>>>>>vvvvvv
vvvvXXX>>>>>>>>>>>>>>vvvvvv
v>>>vvXX>>>>>^X>>>>>>vvvvvv
v>>>>vvXXX>^XX>>>>>>>>vvvvv
>>>>>>>vvXXXX>>>>>>>>>>vvvv
>>>>>>>>>vv>>>>>^XX>>>>vvvv
vX>>>>>>>vvXXX>^XXvXX>>vvvv
vXXX>>>>>>>vXXXX>>>vXXX>>vv
>>>>>>>>>>>>>>>>>>>>>>>>>>G


# Policy Path

In [25]:
def pretty_print_path(world: World, policy: Policy, start: State, goal: State, costs: Dict[str, int]):
    total_cost, visited, node = 0, set(), start
    map = [[TEXT2EMOJI[val] for val in row] for row in world]
    map[goal[1]][goal[0]] = "🎁"
    while node != goal:
        x, y = node[0], node[1]
        visited.add((x, y))

        action = policy[node]
        tile = ACTION2EMOJI.get(action, map[y][x])  # type: ignore

        node = (x + action[0], y + action[1])
        if node in visited or not (0 <= x < cols and 0 <= y < rows and world[y][x] != "x"):
            return

        total_cost += -1 * costs[world[y][x]]
        map[y][x] = tile

    display_emoji_grid(map)
    return total_cost

In [26]:
pretty_print_path(small_world, small_policy, (0, 0), (len(small_world[0]) - 1, len(small_world) - 1), costs)

⏬,🌾,🌾,🌾,🌾,🌾
⏬,🌲,🌲,🌲,🌲,🌾
⏬,🌲,🌲,🌲,🌲,🌾
⏬,🌲,🌲,🌋,🌲,🌾
⏬,🌲,🌲,🌲,🌲,🌾
⏩,⏩,⏩,⏩,⏬,🌾
🌾,🌾,🌾,🌾,⏩,🎁


11

In [27]:
pretty_print_path(large_world, large_policy, (0, 0), (len(large_world[0]) - 1, len(large_world) - 1), costs)

⏬,🌾,🌾,🌾,🌾,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾
⏬,🌾,🌾,🌾,🌾,🌾,🌾,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌾,🌾,🌋,🌋,🌋,🌋,🌋,🌋,🌋,🌾,🌾
⏬,🌾,🌾,🌾,🌋,🌋,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌋,🌋,🌋,⛰,⛰,⛰,🌋,🌋,⛰,⛰
⏬,🌾,🌾,🌾,⛰,🌋,🌋,🌋,🌲,🌲,🌲,🌲,🐊,🐊,🌲,🌲,🌲,🌲,🌲,🌾,🌾,⛰,⛰,🌋,🌋,⛰,🌾
⏬,🌾,🌾,⛰,⛰,🌋,🌋,🌲,🌲,🌾,🌾,🐊,🐊,🐊,🐊,🌲,🌲,🌲,🌾,🌾,🌾,⛰,🌋,🌋,🌋,⛰,🌾
⏬,⛰,⛰,⛰,🌋,🌋,⛰,⛰,🌾,🌾,🌾,🌾,🐊,🐊,🐊,🐊,🐊,🌾,🌾,🌾,🌾,🌾,⛰,🌋,⛰,🌾,🌾
⏬,⛰,⛰,🌋,🌋,⛰,⛰,🌾,🌾,🌾,🌾,⛰,🌋,🌋,🌋,🐊,🐊,🐊,🌾,🌾,🌾,🌾,🌾,⛰,🌾,🌾,🌾
⏬,🌾,⛰,⛰,⛰,⛰,⛰,🌾,🌾,🌾,🌾,🌾,🌾,⛰,🌋,🌋,🌋,🐊,🐊,🐊,🌾,🌾,⛰,⛰,⛰,🌾,🌾
⏬,🌾,🌾,⛰,⛰,⛰,🌾,🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌋,🌋,🌾,🐊,🐊,🌾,🌾,⛰,⛰,⛰,🌾,🌾,🌾
⏬,🌾,🌾,🐊,🐊,🐊,🌾,🌾,⛰,⛰,⛰,🌋,🌋,🌋,🌋,🌾,🌾,🌾,🐊,🌾,⛰,⛰,⛰,🌾,🌾,🌾,🌾
⏬,🌾,🐊,🐊,🐊,🐊,🐊,🌾,⛰,⛰,🌋,🌋,🌋,⛰,🌾,🌾,🌾,🌾,🌾,⛰,🌋,🌋,🌋,⛰,🌾,🌾,🌾


174

# Comparison to A\*

In [28]:
goal = {(len(astar_small_world[0]) - 1, len(astar_small_world) - 1): 100.0}
gamma = 0.9


astart_small_policy = value_iteration(astar_small_world, costs, goal, cardinal_moves, gamma, transition=0.7)
small_path_cost = pretty_print_path(astar_small_world, astart_small_policy, (0, 0), (len(astar_small_world[0]) - 1, len(astar_small_world) - 1), costs)
print(f"total path cost: {small_path_cost}")

⏬,🌲,🌲,🌲,🌲,🌲,🌲
⏬,🌲,🌲,🌲,🌲,🌲,🌲
⏬,🌲,🌲,🌲,🌲,🌲,🌲
⏩,⏩,⏩,⏩,⏩,⏩,⏬
🌲,🌲,🌲,🌲,🌲,🌲,⏬
🌲,🌲,🌲,🌲,🌲,🌲,⏬
🌲,🌲,🌲,🌲,🌲,🌲,🎁


total path cost: 12


In [29]:
cols = len(astar_full_world[0])
rows = len(astar_full_world)
goal = {(cols - 1, rows - 1): 100.0 * cols * rows}
gamma = 0.9

transition = 0.7
print("Transition = 0.7")
astar_full_policy = value_iteration(astar_full_world, costs, goal, cardinal_moves, gamma, transition, max_iters=-1)
path_cost = pretty_print_path(astar_full_world, astar_full_policy, (0, 0), (cols - 1, rows - 1), costs)
print(f"total path cost: {path_cost}")

print("\n")
print("Transition = 1.0")
transition = 1.0
astar_full_policy = value_iteration(astar_full_world, costs, goal, cardinal_moves, gamma, transition, max_iters=-1)
path_cost = pretty_print_path(astar_full_world, astar_full_policy, (0, 0), (cols - 1, rows - 1), costs)
print(f"total path cost: {path_cost}")

Transition = 0.7


⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏬,🐊,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,⛰,⛰,⛰
🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌾,🌾,🌾,🌾,⏬,🐊,🐊,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,⛰,🌋,🌋,⛰
🌾,🌾,🌾,🌾,🌾,⛰,⛰,⛰,🌾,🌾,🐊,⏬,🐊,🐊,🌾,🌾,🌋,🌾,🌾,🌾,⛰,🌾,🌾,⛰,⛰,🌋,🌾
🌾,🌾,🌾,🌾,⛰,⛰,🌋,⛰,⛰,🐊,🐊,⏬,🐊,🐊,🌾,🌋,🌋,🌋,🌋,🌾,⛰,🌾,🌾,🌾,⛰,🌋,🌾
🌾,🌾,🌋,⛰,⛰,🌋,🌋,⛰,⛰,🐊,🐊,⏬,🐊,🐊,🌋,🌋,🌲,🌋,🌋,🌋,⛰,⛰,🌾,🌾,⛰,⛰,🌾
🌲,🌾,🌋,🌋,🌋,🌋,⛰,⛰,⛰,🐊,🐊,⏩,⏬,🌾,🌾,🌋,🌲,🌲,🌋,🌋,⛰,⛰,🌾,⛰,⛰,🌾,🌾
🌲,🌾,🌲,🌋,🌋,⛰,⛰,⛰,🌾,🌾,🐊,🌾,⏩,⏩,⏩,⏬,🌲,🌲,🌲,🌋,🌋,⛰,⛰,⛰,🌾,🌾,🌾
🌲,🌲,🌲,🌋,🌲,⛰,🌾,🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌲,⏬,🌲,🌲,🌲,🌲,🌋,🌋,⛰,⛰,🌾,🌾,🌾
🌲,🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,⛰,⛰,⛰,⛰,🌲,🌲,⏬,🌲,🌲,🌲,🌲,🌲,🌋,⛰,⛰,🌾,🌾,🌾
🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌋,🌋,🌲,🌲,⏬,🌲,🌲,🌲,🌲,🌲,🌋,🌋,⛰,🌾,🌾,🌾
🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,🌾,⛰,🌋,🌋,🌋,⛰,🌲,⏩,⏬,🌲,🌲,🌲,🌲,🌲,🌋,🌾,🌾,⛰,🌾


total path cost: 174


Transition = 1.0


⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏩,⏬,🌾,⛰,⛰,⛰
🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌾,🌾,🌾,🌾,🐊,🐊,🐊,🌾,🌾,🌾,🌾,🌾,🌾,🌾,🌾,⏬,⛰,🌋,🌋,⛰
🌾,🌾,🌾,🌾,🌾,⛰,⛰,⛰,🌾,🌾,🐊,🐊,🐊,🐊,🌾,🌾,🌋,🌾,🌾,🌾,⛰,🌾,⏬,⛰,⛰,🌋,🌾
🌾,🌾,🌾,🌾,⛰,⛰,🌋,⛰,⛰,🐊,🐊,🐊,🐊,🐊,🌾,🌋,🌋,🌋,🌋,🌾,⛰,🌾,⏩,⏬,⛰,🌋,🌾
🌾,🌾,🌋,⛰,⛰,🌋,🌋,⛰,⛰,🐊,🐊,🐊,🐊,🐊,🌋,🌋,🌲,🌋,🌋,🌋,⛰,⛰,🌾,⏩,⏩,⏩,⏬
🌲,🌾,🌋,🌋,🌋,🌋,⛰,⛰,⛰,🐊,🐊,🐊,🌾,🌾,🌾,🌋,🌲,🌲,🌋,🌋,⛰,⛰,🌾,⛰,⛰,🌾,⏬
🌲,🌾,🌲,🌋,🌋,⛰,⛰,⛰,🌾,🌾,🐊,🌾,🌾,🌾,🌾,🌲,🌲,🌲,🌲,🌋,🌋,⛰,⛰,⛰,🌾,🌾,⏬
🌲,🌲,🌲,🌋,🌲,⛰,🌾,🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌲,🌲,🌲,🌲,🌲,🌲,🌋,🌋,⛰,⛰,🌾,🌾,⏬
🌲,🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,⛰,⛰,⛰,⛰,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌋,⛰,⛰,🌾,🌾,⏬
🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,🌾,⛰,⛰,🌋,🌋,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌋,🌋,⛰,🌾,🌾,⏬
🌲,🌲,🌲,🌲,🌾,🌾,🌾,🌾,🌾,⛰,🌋,🌋,🌋,⛰,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌲,🌋,🌾,🌾,⛰,⏬


total path cost: 98


# Discussion
- reward 

- comparison to A\* and Q-Learning

- checking bad policy


## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.